In [32]:
import pandas as pd
import numpy as np

In [33]:
df=pd.read_csv("/Users/nidagallusandeep/Downloads/day27_pandas_groupby.csv")

In [34]:
df.head()

,Order_ID,City,Product,Category,Quantity,Unit_Price_INR,Order_Status,Revenue_INR
0,6001,Hyderabad,Laptop,Electronics,1,55000,Delivered,55000
1,6002,Bengaluru,Mouse,Accessories,2,800,Delivered,1600
2,6003,Chennai,Monitor,Electronics,1,12500,Returned,12500
3,6004,Pune,Keyboard,Accessories,1,1500,Delivered,1500
4,6005,Hyderabad,Laptop,Electronics,1,62000,Cancelled,62000


In [35]:
df.shape

(30, 8)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Order_ID        30 non-null     int64 
 1   City            30 non-null     object
 2   Product         30 non-null     object
 3   Category        30 non-null     object
 4   Quantity        30 non-null     int64 
 5   Unit_Price_INR  30 non-null     int64 
 6   Order_Status    30 non-null     object
 7   Revenue_INR     30 non-null     int64 
dtypes: int64(4), object(4)
memory usage: 2.0+ KB


In [37]:
df.columns

Index(['Order_ID', 'City', 'Product', 'Category', 'Quantity', 'Unit_Price_INR',
       'Order_Status', 'Revenue_INR'],
      dtype='object')

In [38]:
df.describe()

,Order_ID,Quantity,Unit_Price_INR,Revenue_INR
count,30.000000,30.000000,30.000000,30.000000
mean,6015.500000,1.666667,25200.000000,26551.666667
std,8.803408,1.028334,27061.969116,26297.969252
min,6001.000000,1.000000,650.000000,1500.000000
25%,6008.250000,1.000000,1412.500000,2825.000000
50%,6015.500000,1.000000,12250.000000,12750.000000
75%,6022.750000,2.000000,57750.000000,57750.000000
max,6030.000000,5.000000,64000.000000,64000.000000


In [39]:
df.duplicated().sum()

np.int64(0)

In [40]:
df.isnull().sum()

Order_ID          0
City              0
Product           0
Category          0
Quantity          0
Unit_Price_INR    0
Order_Status      0
Revenue_INR       0
dtype: int64

# What is groupby()
- groupby() groups rows that share the same value, so we can calculate statistics for each group.

# The basic syntax:
df.groupby("Column_Name")["Numeric_Column"].sum()

In [41]:
# Total revenue for each city
total_revenue=df.groupby("City")["Revenue_INR"].sum()
total_revenue

City
Bengaluru    227400
Chennai      152550
Hyderabad    205400
Pune         211200
Name: Revenue_INR, dtype: int64

In [42]:
# Count orders by city
count_orders=df.groupby("City")["Order_ID"].count()
count_orders

City
Bengaluru    8
Chennai      7
Hyderabad    8
Pune         7
Name: Order_ID, dtype: int64

In [43]:
# Average revenue by city
avg_revenue=df.groupby("City")["Revenue_INR"].mean()
avg_revenue

City
Bengaluru    28425.000000
Chennai      21792.857143
Hyderabad    25675.000000
Pune         30171.428571
Name: Revenue_INR, dtype: float64

In [44]:
#Total quantity sold for each product
total_quantity=df.groupby("Product")["Quantity"].sum()
total_quantity

Product
Keyboard    11
Laptop      11
Monitor      9
Mouse       19
Name: Quantity, dtype: int64

In [45]:
# Revenue by product, highest first
product_sales=df.groupby("Product")["Revenue_INR"].max().sort_values(ascending=False)
product_sales

Product
Laptop      64000
Monitor     24000
Keyboard     4050
Mouse        3250
Name: Revenue_INR, dtype: int64

- To display the highest revenue first, use ascending=False.

# Group by multiple columns

In [49]:
#  Revenue by city and product
result=df.groupby(["City", "Product"])["Revenue_INR"].sum().head()
result

City       Product 
Bengaluru  Keyboard      3200
           Laptop      184500
           Monitor      36300
           Mouse         3400
Chennai    Keyboard      4050
Name: Revenue_INR, dtype: int64

- This shows revenue for each city-product combination.

- For example, Hyderabad Laptop revenue and Bengaluru Laptop revenue appear as separate groups

In [53]:
# Revenue by city and order status
result=df.groupby(["City","Order_Status"])["Revenue_INR"].sum().sort_values(ascending=False)
result

City       Order_Status
Bengaluru  Delivered       212900
Pune       Delivered       149400
Hyderabad  Delivered       140600
Chennai    Delivered        83050
           Returned         69500
Hyderabad  Cancelled        62000
Pune       Cancelled        61800
Bengaluru  Cancelled        12700
Hyderabad  Returned          2800
Bengaluru  Returned          1800
Name: Revenue_INR, dtype: int64

In [54]:
# Use multiple aggregation functions with agg()
result = df.groupby("Product").agg(
    Total_Quantity=("Quantity", "sum"),
    Total_Revenue=("Revenue_INR", "sum"),
    Average_Revenue=("Revenue_INR", "mean"),
    Order_Count=("Order_ID", "count"))
result

,Total_Quantity,Total_Revenue,Average_Revenue,Order_Count
Product,,,,
Keyboard,11,16000,2666.666667,6
Laptop,11,656000,59636.363636,11
Monitor,9,110300,15757.142857,7
Mouse,19,14250,2375.000000,6


- This calculates four metrics for each product in a single operation.
- sum → total
- mean → average
- count → number of non-missing values

# as_index=False — Keep groups as columns
- By default, the grouped column becomes the index. Use as_index=False when you want a regular DataFrame.

In [55]:
result = df.groupby("City",as_index=False).agg(Total_Revenue=("Revenue_INR", "sum"))
print(result)

        City  Total_Revenue
0  Bengaluru         227400
1    Chennai         152550
2  Hyderabad         205400
3       Pune         211200


# Filter groups with filter()

In [59]:
# Find products whose total quantity sold is greater than 5:
result = df.groupby("Product").filter(lambda group: group["Quantity"].sum() > 5).head()
result

,Order_ID,City,Product,Category,Quantity,Unit_Price_INR,Order_Status,Revenue_INR
0,6001,Hyderabad,Laptop,Electronics,1,55000,Delivered,55000
1,6002,Bengaluru,Mouse,Accessories,2,800,Delivered,1600
2,6003,Chennai,Monitor,Electronics,1,12500,Returned,12500
3,6004,Pune,Keyboard,Accessories,1,1500,Delivered,1500
4,6005,Hyderabad,Laptop,Electronics,1,62000,Cancelled,62000


In [63]:
# Average unit price by product
df.groupby("Product")["Unit_Price_INR"].mean()

Product
Keyboard     1475.000000
Laptop      59636.363636
Monitor     12357.142857
Mouse         775.000000
Name: Unit_Price_INR, dtype: float64

In [65]:
# Maximum unit price for each product
df.groupby("Product")["Unit_Price_INR"].max()

Product
Keyboard     1600
Laptop      64000
Monitor     13200
Mouse         900
Name: Unit_Price_INR, dtype: int64

In [66]:
# Minimum unit price for each product
df.groupby("Product")["Unit_Price_INR"].min()

Product
Keyboard     1350
Laptop      55000
Monitor     11500
Mouse         650
Name: Unit_Price_INR, dtype: int64

In [68]:
# Average quantity by city
avg_quantity=df.groupby("City")["Quantity"].mean()
avg_quantity

City
Bengaluru    1.500000
Chennai      1.857143
Hyderabad    1.625000
Pune         1.714286
Name: Quantity, dtype: float64

In [69]:
# Order count by city and status
df.groupby(["City","Order_Status"])["Order_ID"].count()

City       Order_Status
Bengaluru  Cancelled       1
           Delivered       6
           Returned        1
Chennai    Delivered       5
           Returned        2
Hyderabad  Cancelled       1
           Delivered       6
           Returned        1
Pune       Cancelled       2
           Delivered       5
Name: Order_ID, dtype: int64

In [71]:
# Total quantity by category and product
total_quantity=df.groupby(["Product","Category"])["Unit_Price_INR"].sum()
total_quantity

Product   Category   
Keyboard  Accessories      8850
Laptop    Electronics    656000
Monitor   Electronics     86500
Mouse     Accessories      4650
Name: Unit_Price_INR, dtype: int64

In [72]:
# Revenue by product, highest first
df.groupby("Product")["Revenue_INR"].sum().sort_values(ascending=False)

Product
Laptop      656000
Monitor     110300
Keyboard     16000
Mouse        14250
Name: Revenue_INR, dtype: int64

In [73]:
# Count unique products in each city
df.groupby("City")["Product"].nunique()

City
Bengaluru    4
Chennai      4
Hyderabad    4
Pune         4
Name: Product, dtype: int64

In [74]:
# Multiple aggregations by city
df.groupby("City").agg(
    Total_Revenue=("Revenue_INR", "sum"),
    Average_Revenue=("Revenue_INR", "mean"),
    Total_Orders=("Order_ID", "count"))

,Total_Revenue,Average_Revenue,Total_Orders
City,,,
Bengaluru,227400,28425.000000,8
Chennai,152550,21792.857143,7
Hyderabad,205400,25675.000000,8
Pune,211200,30171.428571,7


In [75]:
# Group by city and keep city as a regular column
df.groupby("City", as_index=False).agg(
    Total_Revenue=("Revenue_INR", "sum"))

,City,Total_Revenue
0,Bengaluru,227400
1,Chennai,152550
2,Hyderabad,205400
3,Pune,211200


In [76]:
# Calculate each city's percentage of total revenue
city_revenue = df.groupby("City")["Revenue_INR"].sum()
city_percentage = (city_revenue / city_revenue.sum()) * 100
print(city_percentage.round(2))

City
Bengaluru    28.55
Chennai      19.15
Hyderabad    25.79
Pune         26.51
Name: Revenue_INR, dtype: float64
